# Recurrent Neural Networks — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/rnn/rnn-lab.ipynb)

Companion notebook for the **Recurrent Neural Networks** track (`rnn-m1` …
`rnn-m14`). Every number the modules quote — the parameter-count formulas,
the BPTT gradient-decay estimate, the LSTM gate count — is re-derived here and
checked with `assert`. The centrepiece is the **parity task**: does a binary
sequence contain an even or odd number of 1s. It needs the *entire* sequence
to answer (flip one early bit and the label flips), which makes it a clean,
fast, honest stress test for exactly the long-range-dependency problem
Modules 8-10 are about.

Everything runs on **CPU in well under a minute total**. No dataset downloads.

| Part | Modules | What runs |
|---|---|---|
| 1 | rnn-m5 – rnn-m7 | the cell equation, parameter-count formula, shared weights |
| 2 | rnn-m8 – rnn-m9 | BPTT gradient product, vanishing/exploding, gradient clipping |
| 3 | rnn-m10 | LSTM/GRU gate parameter counts, and the parity task itself |
| 4 | rnn-m11 – rnn-m12 | padding/packing variable-length batches, the four sequence patterns |

## Setup

In [ ]:
import importlib.util, subprocess, sys

for pkg, module in [('torch', 'torch'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        print(f'installing {pkg} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    else:
        print(f'{pkg:<12} already present')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import random, math
import matplotlib.pyplot as plt

torch.manual_seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams['figure.figsize'] = (7, 4)
print('torch  ', torch.__version__)
print('device ', DEVICE)

---
# Part 1 — The RNN cell

## rnn-m6 / rnn-m7 · The cell equation, and its parameter count

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b), \qquad \hat y_t = W_y h_t + b_y$$

Three matrices and two biases, **shared across every time step** — the count
below does not depend on sequence length at all.

In [ ]:
class VanillaRNNCell(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.Wx = nn.Linear(d, n, bias=False)
        self.Wh = nn.Linear(n, n, bias=True)   # one bias is enough; PyTorch's nn.RNN uses two

    def forward(self, x, h):
        return torch.tanh(self.Wx(x) + self.Wh(h))

d, n, k = 50, 128, 5
cell = VanillaRNNCell(d, n)
Wy = nn.Linear(n, k)

cell_params = sum(p.numel() for p in cell.parameters())
out_params = sum(p.numel() for p in Wy.parameters())
total = cell_params + out_params
formula = d*n + n*n + n + k*n + k
print(f'cell parameters (Wx, Wh, b): {cell_params:,}')
print(f'output head (Wy, b_y):      {out_params:,}')
print(f'total:                      {total:,}')
print(f'formula nd+n^2+n+kn+k:      {formula:,}')
assert total == formula == 23_557          # the rnn-quiz figure, d=50 n=128 k=5
print('\nThis count is the SAME whether the sequence has 5 steps or 5,000 — that is')
print('rnn-m7\'s whole point: the weights are reused, not duplicated, per step.')

---
# Part 2 — Backpropagation through time

## rnn-m8 / rnn-m9 · The gradient product, and why it vanishes

Unrolled, the gradient reaching step 1 from a loss at step $T$ is a **product**
of $T$ Jacobians. Approximate each Jacobian's effect by a scalar — the
recurrent weight's spectral radius times the tanh derivative at that step —
and the whole chain becomes one number raised to the $T$-th power.

In [ ]:
spectral_radius = 0.9
tanh_deriv = 0.8
T = 10
estimate = 1.0 * (spectral_radius * tanh_deriv) ** T
print(f'per-step factor: {spectral_radius} x {tanh_deriv} = {spectral_radius*tanh_deriv}')
print(f'after {T} steps: {estimate:.4f}   (rnn-quiz: "Approximately 0.037")')
assert abs(estimate - 0.037) < 0.001

# Now measure the REAL thing, not the scalar approximation, on an actual RNN.
def grad_at_first_step(depth, cell_type):
    torch.manual_seed(0)
    n = 32
    if cell_type == 'rnn':
        rnn = nn.RNN(1, n, batch_first=True)
    elif cell_type == 'lstm':
        rnn = nn.LSTM(1, n, batch_first=True)
    else:
        rnn = nn.GRU(1, n, batch_first=True)
    x = torch.randn(4, depth, 1, requires_grad=True)
    out, _ = rnn(x) if cell_type != 'lstm' else rnn(x)
    out[:, -1].sum().backward()
    return x.grad[:, 0].abs().mean().item()      # gradient that reached the FIRST input

print(f'\n{"depth":>6}{"vanilla RNN":>14}{"LSTM":>14}{"GRU":>14}')
for depth in (5, 20, 60, 120):
    r = grad_at_first_step(depth, 'rnn')
    l = grad_at_first_step(depth, 'lstm')
    g = grad_at_first_step(depth, 'gru')
    print(f'{depth:>6}{r:>14.2e}{l:>14.2e}{g:>14.2e}')
print('\nThe vanilla RNN\'s gradient at the first step collapses toward zero as depth')
print('grows — LSTM/GRU decay far more slowly, because their gates let gradient')
print('skip the repeated tanh squashing (rnn-m10).')

## rnn-m9 · Gradient clipping, measured

In [ ]:
rnn = nn.RNN(1, 64, batch_first=True)
x = torch.randn(8, 200, 1) * 3          # a deliberately large-scale input to provoke big gradients
out, _ = rnn(x)
out.sum().backward()

raw_norm = torch.sqrt(sum(p.grad.pow(2).sum() for p in rnn.parameters() if p.grad is not None))
print(f'raw gradient norm before clipping: {raw_norm:.2f}')

torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
clipped_norm = torch.sqrt(sum(p.grad.pow(2).sum() for p in rnn.parameters() if p.grad is not None))
print(f'gradient norm after clip_grad_norm_(max_norm=1.0): {clipped_norm:.4f}')
assert clipped_norm <= 1.0 + 1e-4
print('\nClipping rescales the WHOLE gradient vector so its norm is at most max_norm —')
print('direction is preserved, only magnitude is capped.')

---
# Part 3 — LSTM/GRU, and the parity task

## rnn-m10 · Gate parameter counts

In [ ]:
def gate_params(d, n):
    """One gate: Linear(d, n) for input + Linear(n, n) for hidden + bias."""
    return n*d + n*n + n

d, n = 100, 256
four_gates = 4 * gate_params(d, n)
print(f'one gate:   {gate_params(d, n):,} parameters')
print(f'four gates: {four_gates:,} parameters   (rnn-quiz LSTM figure)')
assert four_gates == 365_568

lstm = nn.LSTM(d, n, batch_first=True)
lstm_actual = sum(p.numel() for p in lstm.parameters())
print(f'nn.LSTM actual parameter count: {lstm_actual:,}')
assert lstm_actual == four_gates      # PyTorch's LSTM: exactly 4 gates, no extra terms

gru = nn.GRU(d, n, batch_first=True)
gru_actual = sum(p.numel() for p in gru.parameters())
print(f'nn.GRU  actual parameter count: {gru_actual:,}   (3 gates, not 4 — {gru_actual/lstm_actual:.0%} of LSTM)')
assert gru_actual == 3 * gate_params(d, n)

## The parity task

Label = 1 if the sequence contains an odd number of 1-bits, else 0. Flipping
any single bit anywhere in the sequence flips the label — there is no
shortcut, the whole sequence must be read to get it right.

In [ ]:
def make_parity_batch(batch_size, seq_len):
    bits = torch.randint(0, 2, (batch_size, seq_len, 1)).float()
    labels = (bits.sum(dim=(1, 2)) % 2).long()
    return bits, labels

class SeqClassifier(nn.Module):
    def __init__(self, cell_type, hidden=32):
        super().__init__()
        Cell = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[cell_type]
        self.rnn = Cell(1, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)

    def forward(self, x):
        out, h = self.rnn(x)
        h_last = h[0] if isinstance(h, tuple) else h    # LSTM returns (h, c)
        return self.out(h_last.squeeze(0))

def train_and_eval(cell_type, seq_len, steps=600):
    torch.manual_seed(0)
    model = SeqClassifier(cell_type).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(steps):
        x, y = make_parity_batch(64, seq_len)
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x, y = make_parity_batch(1000, seq_len)
        x, y = x.to(DEVICE), y.to(DEVICE)
        acc = (model(x).argmax(-1) == y).float().mean().item()
    return acc

print(f'{"length":>8}{"vanilla RNN":>14}{"LSTM":>10}{"GRU":>10}   (chance = 0.500)')
for L in (5, 15, 30, 50):
    r = train_and_eval('rnn', L)
    l = train_and_eval('lstm', L)
    g = train_and_eval('gru', L)
    print(f'{L:>8}{r:>14.3f}{l:>10.3f}{g:>10.3f}')
print('\nAt short lengths all three learn parity fine. As length grows, the vanilla')
print('RNN\'s accuracy degrades toward chance while LSTM/GRU hold up far better —')
print('the vanishing-gradient argument from rnn-m9/rnn-m10, as a measured number.')

---
# Part 4 — Variable length, and the four patterns

## rnn-m11 · Padding and packing

In [ ]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

seqs = [torch.randn(4, 3), torch.randn(2, 3), torch.randn(6, 3)]     # lengths 4, 2, 6
lengths = torch.tensor([s.size(0) for s in seqs])

padded = pad_sequence(seqs, batch_first=True)          # (3, 6, 3) — zero-padded to the longest
print('padded shape:', tuple(padded.shape))

packed = pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
rnn = nn.RNN(3, 8, batch_first=True)
out_packed, h = rnn(packed)
out_padded, out_lengths = pad_packed_sequence(out_packed, batch_first=True)
print('unpacked output shape:', tuple(out_padded.shape))
print('recovered lengths:', out_lengths.tolist())
assert out_lengths.tolist() == lengths.tolist()
print('\npack_padded_sequence tells the RNN the REAL length of each sequence, so it')
print('skips computing over the padding — no wasted compute, no polluted hidden state.')

## rnn-m12 · The four sequence patterns, in one cell each

In [ ]:
hidden = 16
rnn = nn.RNN(3, hidden, batch_first=True)
x = torch.randn(2, 5, 3)          # batch=2, T=5 steps, input dim 3
out, h_final = rnn(x)

print('many-to-one   (e.g. sentiment):  use h_final only        ->', tuple(h_final.shape))
print('one-to-many   (e.g. captioning): feed h_final, unroll T\' new steps -> (batch, T\', hidden)')
print('many-to-many, synced (e.g. NER): use every out[:, t]     ->', tuple(out.shape))
print('many-to-many, encoder-decoder:   h_final becomes ANOTHER RNN\'s h_0 ->', tuple(h_final.shape))
print('\nSame nn.RNN call underneath every pattern — what differs is only which')
print('outputs you keep and what you do with them afterward.')

---
## Where to go next

- **Attention.** rnn-m14 ends on the encoder-decoder bottleneck — the single
  fixed context vector that has to represent an entire source sequence. The
  **Attention & Encoder-Decoder Models** track picks up exactly there, with
  its own notebook measuring the same bottleneck directly.
- **Bidirectionality.** Every `nn.RNN`/`nn.LSTM`/`nn.GRU` call above takes
  `bidirectional=True` for free — worth comparing against rnn-m14's BiLSTM NER
  claim on a real tagging dataset.
- **Deeper stacks.** `num_layers=2` or `3` on any of the cells above, and
  re-run the parity sweep — depth interacts with the vanishing-gradient
  picture in its own way, layered on top of sequence length.

Re-run with a bigger `hidden` and more `steps` and the parity numbers above
improve — but the *shape* of vanilla-RNN-vs-LSTM/GRU gap will not change.